# 02 — Feature Engineering: Lags, Rolling Stats, and the Leakage Trap

Goal: understand why `features.py` builds lag/rolling features the specific way it
does — especially the `SHIFT`/`PRECEDING` logic — by breaking it on purpose first,
then fixing it.

We'll work on ONE series (one item at one store) so every number is checkable by eye.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR = Path("../data/raw")
pd.set_option("display.max_columns", 15)

# build one clean series manually, same as notebook 01 did
sales_raw = pd.read_csv(RAW_DIR / "sales_train_validation.csv", nrows=1)
calendar = pd.read_csv(RAW_DIR / "calendar.csv")

day_cols = [c for c in sales_raw.columns if c.startswith("d_")][:30]  # just first 30 days
one_series = sales_raw[['item_id','store_id'] + day_cols].melt(
    id_vars=['item_id','store_id'], var_name='d', value_name='sales'
)
one_series = one_series.merge(calendar[['d','date']], on='d').sort_values('date').reset_index(drop=True)
one_series[['date','sales']]


,date,sales
0,2011-01-29,0
1,2011-01-30,0
2,2011-01-31,0
3,2011-02-01,0
4,2011-02-02,0
5,2011-02-03,0
6,2011-02-04,0
7,2011-02-05,0
8,2011-02-06,0
9,2011-02-07,0


## Step 1 — What is a "lag" feature, mechanically?

`lag_7` for a given day should answer: "what were sales 7 days before this day?"
It is NOT "what are sales 7 days from now" — direction matters. Let's build it by
hand with pandas `.shift()`.

In [3]:
s = one_series.copy()
s['lag_1'] = s['sales'].shift(1)   # yesterday's sales
s['lag_7'] = s['sales'].shift(7)   # sales 7 days ago
s[['date','sales','lag_1','lag_7']].head(12)


,date,sales,lag_1,lag_7
0,2011-01-29,0,NaN,NaN
1,2011-01-30,0,0.0,NaN
2,2011-01-31,0,0.0,NaN
3,2011-02-01,0,0.0,NaN
4,2011-02-02,0,0.0,NaN
5,2011-02-03,0,0.0,NaN
6,2011-02-04,0,0.0,NaN
7,2011-02-05,0,0.0,0.0
8,2011-02-06,0,0.0,0.0
9,2011-02-07,0,0.0,0.0


Check row by row: `lag_1` on any date should equal `sales` from the PREVIOUS row.
`lag_7` should equal `sales` from 7 rows above. The first 7 rows have `NaN` for
`lag_7` — there's no history yet for those days. This is expected and correct,
not a bug. (This is also why `features.py` later drops rows where the largest
lag is still NULL — those rows don't have enough history to be usable.)

## Step 2 — Now let's build a rolling mean, and deliberately get it WRONG first

The naive way most people first try: just call `.rolling(7).mean()` directly on
`sales`. Let's see what's wrong with that.

In [4]:
s['roll_mean_7_WRONG'] = s['sales'].rolling(7).mean()
s[['date','sales','roll_mean_7_WRONG']].head(10)


,date,sales,roll_mean_7_WRONG
0,2011-01-29,0,NaN
1,2011-01-30,0,NaN
2,2011-01-31,0,NaN
3,2011-02-01,0,NaN
4,2011-02-02,0,NaN
5,2011-02-03,0,NaN
6,2011-02-04,0,0.0
7,2011-02-05,0,0.0
8,2011-02-06,0,0.0
9,2011-02-07,0,0.0


**Look closely at any row.** `roll_mean_7_WRONG` on a given date INCLUDES that
day's own `sales` value in the average. That means: if you used this as a feature
to predict `sales` on that same day, the feature literally contains the answer.

This is **leakage** — the model would see a feature that's partly made of the exact
number it's trying to predict. During training this often looks GREAT (suspiciously
good validation metrics), and then falls apart on live/future data, because at
prediction time you don't actually know today's sales yet — that's the whole point
of forecasting it.

## Step 3 — the fix: shift by 1 before rolling

We want "the average of the last 7 days BEFORE today", not "the last 7 days
including today". So shift the series by 1 first, THEN roll.

In [5]:
s['roll_mean_7_correct'] = s['sales'].shift(1).rolling(7).mean()
s[['date','sales','roll_mean_7_WRONG','roll_mean_7_correct']].head(10)


,date,sales,roll_mean_7_WRONG,roll_mean_7_correct
0,2011-01-29,0,NaN,NaN
1,2011-01-30,0,NaN,NaN
2,2011-01-31,0,NaN,NaN
3,2011-02-01,0,NaN,NaN
4,2011-02-02,0,NaN,NaN
5,2011-02-03,0,NaN,NaN
6,2011-02-04,0,0.0,NaN
7,2011-02-05,0,0.0,0.0
8,2011-02-06,0,0.0,0.0
9,2011-02-07,0,0.0,0.0


Now `roll_mean_7_correct` on any date is computed ONLY from the 7 days strictly
before it. Compare the two columns — they're offset by one day. That one-day
offset is the entire difference between a valid feature and a leaky one.

This is exactly what `features.py` does in SQL, using window frame syntax instead
of `.shift()`:

```sql
AVG(sales) OVER (
    PARTITION BY item_id, store_id ORDER BY date
    ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
) AS roll_mean_7
```

`ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING` means "the 7 rows before this one, NOT
including this one" — same logic as `shift(1).rolling(7)`, just expressed as a SQL
window frame instead of pandas.

## Step 4 — same leakage trap applies to `LAG`, just less obviously

`LAG(sales, 7)` is already safe by construction — it only ever looks backward, never
at the current row. But it's worth noticing: **every single feature in this project
must only use information that would have been known BEFORE that day actually
happened.** That's the real rule. Lags and shifted rolling stats are two ways of
satisfying it; calendar features (day-of-week, month) automatically satisfy it too,
since a date is known in advance. Price is a bit more subtle — worth thinking about
on your own: is today's `sell_price` safe to use as-is, or could it leak?

(Hint: is price something you'd know before the day happens, or only after?)

## Step 5 — rolling std, and why it matters later for anomaly detection

`features.py` also computes `roll_std_7` and `roll_std_28` — rolling standard
deviation, same shift-first logic. This measures how *volatile* a series has been
recently, not just its average level.

In [6]:
s['roll_std_7_correct'] = s['sales'].shift(1).rolling(7).std()
s[['date','sales','roll_mean_7_correct','roll_std_7_correct']].head(12)


,date,sales,roll_mean_7_correct,roll_std_7_correct
0,2011-01-29,0,NaN,NaN
1,2011-01-30,0,NaN,NaN
2,2011-01-31,0,NaN,NaN
3,2011-02-01,0,NaN,NaN
4,2011-02-02,0,NaN,NaN
5,2011-02-03,0,NaN,NaN
6,2011-02-04,0,NaN,NaN
7,2011-02-05,0,0.0,0.0
8,2011-02-06,0,0.0,0.0
9,2011-02-07,0,0.0,0.0


Why does this matter? Two series can have the same rolling mean but wildly
different rolling std — one is steady, one is spiky. A model (and later, the
anomaly detector in notebook 05) needs to know the difference: a big jump in a
naturally spiky series might be normal, while the same-sized jump in a steady
series is a real anomaly. Raw residuals alone can't tell those apart — you need
the volatility context. Keep this in mind, it comes back directly in notebook 05.

## Check your understanding

1. Why does `roll_mean_7_WRONG` produce a leaky feature, in your own words?
2. What's the one-line fix that turns a leaky rolling feature into a safe one?
3. Why do the first few rows of `lag_7` end up `NaN`, and why is that expected
   rather than something to patch?
4. Is `sell_price` on a given day safe to use as a feature for predicting that
   day's sales, or does it need the same shift treatment as sales? Reason about
   when a retailer actually sets a price vs when they observe demand.

Once these are solid, move to notebook 03 — the point forecast model, and why
one global LightGBM model handles all 30,490 series.